# Notebook 06: RAG Explorer

Chatbot Gradio + analisis interactivo del engine: busqueda, diagnostico, thumbnails, audio, notas, cross-session, timeline.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('/Users/alexmartin/Documents/video')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag.query_engine import NotebookRAGEngine, format_timestamp

engine = NotebookRAGEngine()
ollama_ok = engine.check_ollama()

print(f'Sessions: {engine.get_session_ids()}')
print(f'FAISS docs: {engine.faiss_enriched.ntotal}')
print(f'Ollama: {"OK" if ollama_ok else "NO"}')

## 1. Chatbot Gradio
Lanza la interfaz completa en localhost:7860

In [ ]:
from src.rag.chat_ui import build_ui

demo = build_ui()
demo.launch(server_port=7860, inline=False)

## 2. Búsqueda directa (sin LLM)
Hybrid FAISS+BM25 — ver qué chunks recupera el engine

In [ ]:
query = "qué es fine-tuning"

results = engine.search(query, top_k=8)

for i, r in enumerate(results, 1):
    ts = format_timestamp(r.get('start_ms', 0))
    session = r.get('session_id', '?')
    topic = r.get('topic', '—')
    score = r.get('rerank_score', r.get('score', 0))
    preview = r.get('content', r.get('text', ''))[:120]
    print(f'{i}. [{session}] {ts} — {topic} (score: {score:.3f})')
    print(f'   {preview}...')
    print()

## 3. Pregunta con LLM (retrieval + generación)
Modos: `explicar`, `tecnico`, `examen`, `resumen`, `apuntes`

In [ ]:
from IPython.display import Markdown, display

query = "qué es LoRA y para qué sirve?"
mode = "explicar"

result = engine.ask(query, mode=mode, top_k=5)

display(Markdown(f'### Respuesta ({mode})\n\n{result["response"]}'))

print(f'\n--- Fuentes ---')
for s in result['sources']:
    print(f"  [{s['session_id']}] {s['timestamp']} — {s['topic']} (q:{s['quality']}, score:{s['score']})")

t = result['timing']
print(f'\n--- Timing: retrieval {t.get("retrieval_s","?")}s | LLM {t.get("llm_s","?")}s | total {t.get("total_s","?")}s ---')

## 4. Diagnóstico
Compara retrieval vs respuesta LLM — detecta alucinaciones

In [ ]:
query = "qué es LoRA y para qué sirve?"

print(engine.diagnose(query, top_k=5))

## 5. Búsqueda cross-session
¿En qué clases se habla de un tema?

In [ ]:
query = "embeddings"

result = engine.cross_session_search(query, top_k=20)

print(f'Total hits: {result["total_hits"]}')
for s in result['sessions']:
    topics = ', '.join(s['topics'][:5])
    print(f"  {s['session_id']}: {s['count']} hits — rango {s['time_range']} — score avg {s['avg_score']}")
    print(f"    Temas: {topics}")

## 6. Apuntes de sesión completa

In [ ]:
from IPython.display import Markdown, display

session_id = "llm_40"

notes = engine.generate_session_notes(session_id, with_llm=ollama_ok)
display(Markdown(notes))

## 7. Resumen en una frase de una sesión
Genera los apuntes completos y pide a Ollama que resuma en una frase de qué se habló

In [ ]:
import requests
from IPython.display import Markdown, display

session_id = "llm_40"

# 1. Generar apuntes completos
notes = engine.generate_session_notes(session_id, with_llm=ollama_ok)

# 2. Pedir a Ollama un resumen en una frase
resp = requests.post(
    f"{engine.ollama_url}/api/chat",
    json={
        "model": engine.ollama_model,
        "messages": [
            {"role": "system", "content": "Eres un asistente que resume clases universitarias. Responde SOLO con una frase concisa."},
            {"role": "user", "content": f"Resume en UNA sola frase de qué se habla en esta sesión de clase:\n\n{notes}"},
        ],
        "stream": False,
        "options": {"temperature": 0.0, "num_predict": 150},
    },
    timeout=30,
)
summary = resp.json()["message"]["content"].strip()

display(Markdown(f"### {session_id} — Resumen\n\n> {summary}"))
print(f"\n--- Apuntes completos usados: {len(notes)} chars ---")

## 8. Consulta temporal — ¿qué se dijo en el minuto X?

In [ ]:
from IPython.display import Markdown, display

session_id = "llm_40"
start_min = 20
end_min = 30

result = engine.summarize_session_part(
    session_id, part="custom",
    start_ms=start_min * 60_000,
    end_ms=end_min * 60_000,
    mode="resumen",
)

display(Markdown(f'### {session_id} — {result.get("range","")}'  ))
display(Markdown(result['response']))

for s in result.get('sources', []):
    print(f"  {s['timestamp']} — {s['topic']}")

## 9. Topics de una sesión (ranking)

In [ ]:
session_id = "llm_40"

topics = engine.get_session_topics(session_id, limit=15)

for i, t in enumerate(topics, 1):
    first = format_timestamp(t["first_ms"])
    last = format_timestamp(t["last_ms"])
    print(f'{i:2d}. {t["topic"]} (count: {t["count"]}, avg_quality: {t["avg_quality"]:.2f}, range: {first} — {last})')

## 10. Video + Thumbnails de momentos clave
Reproduce el video de la sesión y extrae frames en los timestamps de la respuesta (ajustado x2)

In [ ]:
import subprocess, base64
from IPython.display import display, HTML, Video

VIDEO_SPEED_FACTOR = 2.0
RECORDINGS_DIR = PROJECT_ROOT / 'recordings' / 'cropped'

def get_video_path(session_id):
    num = session_id.replace('llm_', '')
    for mp4 in RECORDINGS_DIR.glob(f'{num}_*.mp4'):
        return str(mp4.resolve())
    return None

def extract_frame(video_path, class_seconds, out_dir='/tmp/rag_nb_thumbs'):
    """Extrae un frame. class_seconds = tiempo original de clase."""
    Path(out_dir).mkdir(exist_ok=True)
    video_seconds = class_seconds * VIDEO_SPEED_FACTOR
    m, s = divmod(int(class_seconds), 60)
    label = f'{m:02d}:{s:02d}'
    out = Path(out_dir) / f'frame_{label.replace(":","-")}.jpg'
    if not out.exists():
        subprocess.run(
            ['ffmpeg','-y','-ss',str(video_seconds),'-i',video_path,
             '-frames:v','1','-q:v','3','-vf','scale=480:-1',str(out)],
            capture_output=True, timeout=10,
        )
    if out.exists():
        return str(out), label
    return None, label

def img_to_base64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

query = "qué es fine-tuning"
results = engine.search(query, top_k=5)

from collections import defaultdict
by_session = defaultdict(list)
for r in results:
    sid = r.get('session_id', '')
    by_session[sid].append(r)

for sid, chunks in by_session.items():
    vpath = get_video_path(sid)
    if not vpath:
        print(f'\n{sid}: sin video disponible')
        for c in chunks:
            ts = format_timestamp(c.get('start_ms', 0))
            print(f'  {ts} — {c.get("topic", "?")}')
        continue
    
    print(f'\n### {sid}')
    display(Video(vpath, embed=True, width=560))
    
    html_parts = []
    for c in chunks:
        secs = c.get('start_ms', 0) / 1000
        path, label = extract_frame(vpath, secs)
        topic = c.get('topic', '—')
        if path:
            b64 = img_to_base64(path)
            html_parts.append(
                f'<div style="display:inline-block;text-align:center;margin:6px">'
                f'<img src="data:image/jpeg;base64,{b64}" width="240" style="border-radius:6px"/><br/>'
                f'<b>{label}</b> — {topic}</div>'
            )
        else:
            html_parts.append(
                f'<div style="display:inline-block;text-align:center;margin:6px;width:240px">'
                f'<div style="background:#333;color:#fff;height:135px;line-height:135px;border-radius:6px">fuera de rango</div>'
                f'<b>{label}</b> — {topic}</div>'
            )
    display(HTML(''.join(html_parts)))

## 11. Audio clips — escucha fragmentos de las fuentes

In [ ]:
import subprocess
from IPython.display import Audio, display, HTML

VIDEO_SPEED_FACTOR = 2.0

def extract_audio_clip(video_path, class_start_s, class_end_s, out_dir='/tmp/rag_nb_audio'):
    """Extrae clip de audio. Tiempos son de clase original (se ajustan x2)."""
    Path(out_dir).mkdir(exist_ok=True)
    video_start = class_start_s * VIDEO_SPEED_FACTOR
    video_duration = (class_end_s - class_start_s) * VIDEO_SPEED_FACTOR
    ms, ss = divmod(int(class_start_s), 60)
    me, se = divmod(int(class_end_s), 60)
    label = f'{ms:02d}:{ss:02d}-{me:02d}:{se:02d}'
    out = Path(out_dir) / f'clip_{label}.mp3'
    if not out.exists():
        subprocess.run(
            ['ffmpeg','-y','-ss',str(video_start),'-i',video_path,
             '-t',str(video_duration),'-vn','-acodec','libmp3lame','-q:a','4',str(out)],
            capture_output=True, timeout=30,
        )
    return str(out), label

query = "qué es fine-tuning"
results = engine.search(query, top_k=5)

for r in results:
    sid = r.get('session_id', '')
    vpath = get_video_path(sid)
    if not vpath:
        continue
    start_s = r.get('start_ms', 0) / 1000
    end_s = r.get('end_ms', r.get('start_ms', 0) + 30000) / 1000
    clip_path, label = extract_audio_clip(vpath, start_s, end_s)
    topic = r.get('topic', '—')
    score = r.get('rerank_score', r.get('score', 0))
    print(f'[{sid}] {label} — {topic} (score: {score:.3f})')
    display(Audio(clip_path))

## 12. Timeline de sesión — todos los chunks ordenados

In [ ]:
from IPython.display import HTML, display

session_id = "llm_40"

chunks = engine.get_session_chunks(session_id)
print(f'{session_id}: {len(chunks)} chunks')

def quality_bar(q):
    pct = int(q * 100)
    color = '#4caf50' if q >= 0.7 else '#ff9800' if q >= 0.5 else '#f44336'
    return f'<div style="background:{color};width:{pct}px;height:12px;border-radius:3px;display:inline-block"></div> {q:.2f}'

rows = []
for c in chunks:
    ts = format_timestamp(c.get('start_ms', 0))
    topic = c.get('topic', '—')
    quality = c.get('quality_score', 0)
    kw = ', '.join(c.get('keywords', [])[:4])
    rows.append(f'<tr><td><code>{ts}</code></td><td>{topic}</td><td>{quality_bar(quality)}</td><td><small>{kw}</small></td></tr>')

html = f'''
<table style="border-collapse:collapse;width:100%">
<tr style="border-bottom:2px solid #333"><th>Tiempo</th><th>Tema</th><th>Calidad</th><th>Keywords</th></tr>
{''.join(rows)}
</table>'''
display(HTML(html))

## 13. Query expansion — ver cómo se expande una query

In [ ]:
query = "qué es fine-tuning"

nq = engine.normalize_query(query)
print(f'Original:   {nq.original}')
print(f'Normalized: {nq.normalized}')
print(f'Core terms: {nq.core_terms}')
print(f'Intent:     {nq.intent}')
print(f'Entities:   {nq.entities}')

expansions = engine.expand_query(nq)
print(f'\nExpansiones ({len(expansions)}):')
for e in expansions:
    print(f'  → {e}')

## 14. Comparar modos — misma pregunta, diferentes estilos

In [ ]:
from IPython.display import Markdown, display

query = "qué es LoRA?"
modes = ["explicar", "tecnico", "examen"]

for mode in modes:
    result = engine.ask(query, mode=mode, top_k=5)
    display(Markdown(f'---\n### Modo: {mode}\n\n{result["response"]}'))
    print(f'  Timing: {result["timing"].get("total_s","?")}s\n')

## 15. Mapa de sesiones — overview de todo el curso

In [ ]:
from IPython.display import HTML, display

rows = []
for sid in engine.get_session_ids():
    chunks = engine.get_session_chunks(sid)
    if not chunks:
        continue
    n = len(chunks)
    start = format_timestamp(chunks[0].get('start_ms', 0))
    end = format_timestamp(chunks[-1].get('end_ms', chunks[-1].get('start_ms', 0)))
    avg_q = sum(c.get('quality_score', 0) for c in chunks) / n if n else 0
    topics = engine.get_session_topics(sid, limit=5)
    top_topics = ', '.join(t['topic'] for t in topics[:5])
    rows.append(
        f'<tr><td><b>{sid}</b></td><td>{n}</td><td>{start} — {end}</td>'
        f'<td>{avg_q:.2f}</td><td><small>{top_topics}</small></td></tr>'
    )

html = f'''
<table style="border-collapse:collapse;width:100%">
<tr style="border-bottom:2px solid #333"><th>Sesión</th><th>Chunks</th><th>Rango</th><th>Calidad avg</th><th>Top temas</th></tr>
{''.join(rows)}
</table>'''
display(HTML(html))